# 02 · Train and compare

Three models (logistic regression, random forest, XGBoost), each trained four ways to
handle the imbalance (about one default in six loans):

| Strategy | What it does to the **training** loans |
| --- | --- |
| `none` | nothing |
| `class_weight` | each default counts as much as all repaid loans together |
| `kmeans_smote` | k-means groups the loans; synthetic defaults are added inside groups where defaults occur |
| `cluster_centroids` | the repaid loans are shrunk to one real loan per k-means group |

**Tests** (decision D-020):

* **Walk forward**, the main test: for each year 2019–2024, train only on loans given
  *and* resolved before that year, then score that year. This is how the Chama would
  use the model.
* **Members held out**, a second check: each member's loans are scored by a model
  that never saw that member.

**Metrics:** PR AUC (are defaults ranked first? chance = the share of defaults), ROC
AUC, and the Brier score (are the scores honest probabilities? lower is better), with
bootstrap 95% intervals. Every run is logged to MLflow with its plots and model.

> The table holds member codes and features only: no names, no free text (decision D-016). The default label is **provisional** until the Chama agrees a definition (D-018): `defaulted` = rolled over while unpaid, overdue, or lost track.

In [ ]:
# Setup: runs in Google Colab and on this computer.
# Colab, once: add a GitHub token that can read this repository as the Colab
# secret GITHUB_TOKEN (key icon on the left). This cell mounts Drive, fetches
# the code and installs what Colab lacks. On this computer it uses DRIVE_DIR
# from .env and the `chama` conda environment.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "m2-features-models"

if IN_COLAB:
    from google.colab import drive, userdata

    REPO = "/content/chama-loan-default"
    PLAIN_URL = "https://github.com/muriithi-alvin/chama-loan-default.git"

    def git(*args):
        """Run git with the token, without ever printing it."""
        url = PLAIN_URL.replace("https://", f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@")
        result = subprocess.run(["git", *[url if a == "URL" else a for a in args]],
                                capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("git failed: check the GITHUB_TOKEN secret and the branch name")

    drive.mount("/content/drive")
    if os.path.exists(REPO):
        git("-C", REPO, "pull", "-q", "URL", BRANCH)
    else:
        git("clone", "-q", "--branch", BRANCH, "URL", REPO)
        git("-C", REPO, "remote", "set-url", "origin", PLAIN_URL)   # no token left behind
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlflow", "shap", "imbalanced-learn", "python-dotenv"], check=True)
    sys.path.insert(0, f"{REPO}/src")
    os.environ["DRIVE_DIR"] = "/content/drive/MyDrive/chama-loan-default"

import warnings

import pandas as pd

from chama.config import export_dir, load_config, mlflow_dir
from chama.features.export import FILE_NAME

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
CONFIG = load_config()
SETTINGS = CONFIG["modelling"]
DATA = export_dir() / FILE_NAME
print("data:  ", DATA)
print("MLflow:", mlflow_dir())

In [ ]:
from chama.models import train

# Takes about 8 minutes for all 12 runs. To try one combination, pass for example
# models=["random_forest"], imbalance=["none", "kmeans_smote"].
summary, scores = train.run(config=CONFIG)
train.summary_table(summary)

## How the risk bands sorted the test loans

Bands come from the score: `low` below 0.15, `medium` from 0.15, `high` from 0.30
(`risk_bands` in `config/pipeline.yaml`, provisional). A useful model puts most
defaults in the high band while keeping its default rate well above the overall share.
Weighting and resampling inflate scores, so their bands fill up: compare the default
*rate* in each band, not only the share of defaults caught.

In [ ]:
from chama.models.evaluate import band_table

best = summary.loc[0, "run"]
print("best walk-forward PR AUC:", best)
band_table(scores[best], SETTINGS["risk_bands"])

In [ ]:
from IPython.display import display

from chama.models import plots

best_kmeans = summary[summary["imbalance"] == "kmeans_smote"].iloc[0]["run"]
for run_name in dict.fromkeys([best, best_kmeans]):
    display(plots.precision_recall(scores[run_name], run_name))
    display(plots.calibration(scores[run_name], run_name))

## Looking at every run in MLflow

On this computer (PowerShell, `chama` environment), with Drive for desktop synced:

```powershell
$env:MLFLOW_ALLOW_FILE_STORE = "true"
mlflow ui --backend-store-uri "file:///G:/My Drive/chama-loan-default/mlruns"
```

Then open http://127.0.0.1:5000, choose the `chama-loan-default` experiment, tick runs
and press **Compare**.